# Welcome to Evidence Graph

**Evidence Graph** is a feature of mercury-graph for building traceable knowledge graphs from text, combining an extraction model
([GLiNER2](https://github.com/fastino-ai/GLiNER2)) with LLM agents.

This notebook takes the point of view of a **library user**: you create an *Endpoint*, start it and use it **from the outside**, through
its HTTP API, exactly as any other client (or agent) would. You will:

1. create the *welcome* Endpoint from the template shipped with the `mge` command line tool,
2. understand its ontology,
3. pilot it and serve it over HTTP,
4. explore its capabilities and navigate its documents and ontologies,
5. extract entities and relationships from its documents,
6. talk to an LLM agent that uses all of the above as tools.

> **Developer preview.** Evidence Graph is under active development: APIs and file formats may change. Section 11 lists what does not
> work yet.

## 0. Configuration

Everything that depends on your machine is in the next cell. The Endpoint folder is created **in the current directory**, and `PORT` is
the HTTP port used from section 5 on.

In [ ]:
import os, sys, time

import pandas as pd
import requests

ENDPOINT_PATH = 'welcome'       # Endpoint folder. Its name can only contain letters, numbers and underscores.
PORT          = 8765            # Port where `mge serve` will listen.
BASE_URL      = 'http://127.0.0.1:%d' % PORT

# The `mge` command is installed next to the Python interpreter that runs this notebook.
MGE = os.path.join(os.path.dirname(sys.executable), 'mge')

# Use litellm's local model cost map instead of downloading it from GitHub.
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'

## 1. The pieces of an Endpoint

| Piece | What it does | In the welcome template |
|---|---|---|
| **Source** | Exposes a corpus of documents as a tree that is navigated level by level. | `md_example`, `mediawiki_example`, `pdf_example` |
| **Ontology** | A graph of concepts: entity types, relationship types, known instances. | `entities`, `relationships`, `known_ids` |
| **Formalizer** | Proposes the entities and relationships found in a text, using the ontologies as labels. | `GLiNER2` |
| **Evidence graph** | Aggregates what is found in the sources into a graph. | `e_graph` *(not functional yet)* |
| **Agent** | An LLM that can call the other pieces as tools. | `reader`, `gru` |

All pieces share the same interface. Each one publishes **capabilities** (functions with a name, a description and arguments) and runs
requests of the form `{"name": ..., "arguments": {...}}`.

The **Endpoint** is a folder that declares all the pieces, connects them to each other and serves the ones marked as *exposed* over HTTP.

## 2. Requirements

**mercury-graph from the repository.** Evidence Graph is not published on PyPI yet, so install it from the `develop` branch:

```bash
git clone https://github.com/BBVA/mercury-graph.git
cd mercury-graph
git checkout develop
python -m venv .venv
source .venv/bin/activate
pip install -e ".[dev]"
```

The `[dev]` extra installs PySpark. For now it is needed: without PySpark, importing `mercury.graph` fails with
`NameError: name 'SparkSession' is not defined` (known issue).

**JupyterLab** in the same environment: `pip install jupyterlab`.

**An LLM for the agents** (only for section 10). The simplest free option is [Ollama](https://ollama.com):

```bash
brew install ollama
brew services start ollama      # or run `ollama serve` in a terminal if you cannot register services
ollama pull qwen3:8b
```

The first time the Endpoint is piloted (section 5), the GLiNER2 model is downloaded from Hugging Face. This can take a few minutes.

The next cell checks your environment.

In [ ]:
import mercury.graph as mg

print('mercury-graph version:', mg.__version__)
print('mge found:', os.path.isfile(MGE))

try:
    models = requests.get('http://localhost:11434/api/tags', timeout = 2).json()['models']
    print('Ollama models:', [m['name'] for m in models])
except requests.ConnectionError:
    print('Ollama is not running. Sections 3 to 9 work without it, section 10 needs it.')

## 3. Create the Endpoint

`mge new` copies the Endpoint template into a new folder. The template is a small, complete example called *welcome*: a fictional school
(Riverside High School, its robotics club and a science fair) described in three document formats.

To start from scratch again, delete the folder and run the cell again.

In [ ]:
if os.path.exists(ENDPOINT_PATH):
    print('The folder "%s" already exists, skipping `mge new`.' % ENDPOINT_PATH)
else:
    !{MGE} new {ENDPOINT_PATH}

The folder contains:

- **`mge_endpoint.jsonc`**, the main configuration file. Each section (`sources`, `ontologies`, `formalizers`, `evidence_graphs`,
  `agents`, `custom_agentics`) lives in its own `.jsonc` file, included with `{"$ref": "./sources.jsonc"}`.
- **`markdown/`, `xml/`, `pdf/`**: the same school documents as Markdown files, as a MediaWiki XML export and as PDF files.
- **`ontology_source/`**: the tab-separated files used to initialize the ontologies.
- **`mge_endpoint.free`**: the lock. It becomes `mge_endpoint.locked` while a process pilots or serves the Endpoint.

`.jsonc` is JSON with comments, with one restriction: **comments must take a whole line** starting with `//`. A comment after a value on
the same line breaks the file.

In [ ]:
for name in sorted(os.listdir(ENDPOINT_PATH)):
    print(name)

This is the main file without its comments. `expose` lists the pieces whose capabilities are published over HTTP.

In [ ]:
with open(os.path.join(ENDPOINT_PATH, 'mge_endpoint.jsonc')) as f:
    print(''.join(line for line in f if not line.strip().startswith('//')))

## 4. Understand the ontology

The Formalizer can only find what the ontologies define. The template ships an ontology for its example documents: the people,
organizations, events, places and projects of a school. There are three ontologies, initialized from the tab-separated files in
`ontology_source/`:

- **`entities`**: entity types, each one with a `definition`.
- **`relationships`**: relationship types, each one with its source type (`src`), destination type (`dest`) and a `definition`.
- **`known_ids`**: concrete, known instances and the edges between them.

In [ ]:
def read_ontology_file(name):
    """ Reads one of the tab-separated files that initialize the ontologies. """

    return pd.read_csv(os.path.join(ENDPOINT_PATH, 'ontology_source', name), sep = '\t')


entities = read_ontology_file('entities_concepts.csv')
entities

In [ ]:
relationships = read_ontology_file('relationships_concepts.csv')
relationships

Identifiers use `|` to build a hierarchy: `person|teacher` is a kind of `person`, and `place|room|laboratory` a kind of `place|room`.
Relationship types can have subtypes too: `works_on|leads`. The hierarchy organizes the ontology for navigation. The Formalizer, however,
sees every identifier as an independent label.

`known_ids` holds instances. Their identifiers start with their type (`person|teacher|Elena Ruiz`), and the edges between them use the
relationship types.

In [ ]:
print(read_ontology_file('known_ids_nodes.csv'))
print()
print(read_ontology_file('known_ids_edges.csv'))

**To change an ontology**, edit its files and keep these rules:

- the files are **tab-separated**,
- parents are written before their children (`person` before `person|teacher`),
- the files are only read when the ontology has not been saved yet. The first pilot saves it in `graphs/<ontology>.pickle`, and from then
  on the saved version wins. To apply new files, stop the server, delete the pickle and pilot again.

## 5. Pilot and serve the Endpoint

**Piloting** takes the Endpoint to a target state. `ALL_READY` (state 100) means that all pieces are loaded, connected to each other and
ready: sources indexed, ontologies loaded and the GLiNER2 model in memory.

`ALL_READY` checks the configuration, not the external services: it does not contact Ollama and it does not convert every document in
advance.

In [ ]:
!{MGE} pilot {ENDPOINT_PATH} ALL_READY

`mge summary` shows every piece of the Endpoint and its state.

In [ ]:
!{MGE} summary {ENDPOINT_PATH}

**Serving** publishes the Endpoint over HTTP. `mge serve` keeps running until you stop it, so it cannot run inside a notebook cell.
Run the command printed below **in a terminal**, using the same Python environment, and keep it open.

While it serves, the Endpoint is locked for other processes. To stop it, press `Ctrl+C` and wait for `Finished server process`: the lock
is released at the very end.

Note that `mge serve` listens on all network interfaces, so the Endpoint is reachable from your network while it runs.

In [ ]:
print('export LITELLM_LOCAL_MODEL_COST_MAP=True')
print('%s serve %s ALL_READY %d' % (MGE, os.path.abspath(ENDPOINT_PATH), PORT))

The next cell waits until the Endpoint answers.

In [ ]:
def wait_for_endpoint(timeout = 300):
    """ Waits until the Endpoint answers and returns its metadata. """

    t0 = time.time()
    while time.time() - t0 < timeout:
        try:
            return requests.get(BASE_URL + '/meta', timeout = 2).json()
        except requests.ConnectionError:
            time.sleep(1)

    raise TimeoutError('No Endpoint is answering at %s' % BASE_URL)


meta = wait_for_endpoint()
print('Endpoint state:', meta['state'])

## 6. The capabilities: `/meta`

`GET /meta` returns the state of the Endpoint and the **capabilities** of all its exposed pieces. Their names follow the pattern
`<function>_<piece name>`.

In [ ]:
for capability in meta['capabilities']:
    function = capability['function']
    print('%-35s %s' % (function['name'], function['parameters'].get('required', [])))

Each capability is described in the **tool format of the OpenAI API**, the same format that [litellm](https://docs.litellm.ai) accepts
to give tools to an LLM. That is why any piece, even another agent, can be used as a tool by an agent: its capabilities are already tool
definitions. The design favors narrow agents that do one thing well and delegate the rest.

The Endpoint also serves interactive API documentation at `/docs`.

Two capabilities are not functional yet: `crawl_e_graph` raises an error, and `is_a_GLiNER2` always returns 0.

In [ ]:
import json

print(json.dumps(meta['capabilities'][0], indent = 2))

## 7. Navigate a Source

`POST /run` runs one capability. This helper sends the request and returns the `message` of the answer. Every answer has the form
`{"finish_reason": ..., "message": ...}`.

In [ ]:
def run(name, **arguments):
    """ Runs a capability of the Endpoint and returns its message. """

    response = requests.post(BASE_URL + '/run', json = {'name': name, 'arguments': arguments}, timeout = 600)
    response.raise_for_status()

    return response.json()['message']

A Source is navigated like a file explorer, **one level at a time**, with two capabilities:

- `children_by_idx_<source>`: the indices below an index ("list this folder"),
- `object_by_idx_<source>`: what is at an index ("open this file").

An index is a path joined with `|`, and it **always starts with the name of the Source**. This is how an agent can explore a corpus much
larger than its context window.

In [ ]:
run('children_by_idx_md_example', index = 'md_example')

Inside a Markdown file, the tree follows its headings: `header_<level>_<n>`. A section contains its paragraphs and its subsections, in
order.

In [ ]:
run('children_by_idx_md_example', index = 'md_example|school.md')

In [ ]:
run('children_by_idx_md_example', index = 'md_example|school.md|header_1_1')

Opening a section returns its title. Paragraphs are containers too: the text itself is in the leaves, `text_<n>`, which are the only nodes
that have a `content`.

In [ ]:
print(run('object_by_idx_md_example', index = 'md_example|school.md|header_1_1|header_2_1'))
print(run('object_by_idx_md_example', index = 'md_example|school.md|header_1_1|paragraph_1'))
print(run('children_by_idx_md_example', index = 'md_example|school.md|header_1_1|paragraph_1'))

In [ ]:
run('object_by_idx_md_example', index = 'md_example|school.md|header_1_1|paragraph_1|text_2')

Mistakes do not raise errors: an index that does not exist returns `None`. A common one is forgetting the name of the Source.

In [ ]:
print(run('children_by_idx_md_example', index = 'school.md'))

With the same two capabilities, a program can walk a whole file. This function collects the text of all the leaves below an index.

A paragraph written over several lines in the file has one leaf per line, which can cut a sentence in two. The function joins the lines
of each paragraph back into a single text.

In [ ]:
def collect_texts(source, index):
    """ Walks the tree of `source` below `index` and returns its texts, in order. The lines of each paragraph are joined into one text. """

    children = run('children_by_idx_%s' % source, index = index)

    if not children:
        leaf = run('object_by_idx_%s' % source, index = index)
        return [leaf['content']] if leaf and 'content' in leaf else []

    texts = []
    for child in children:
        texts.extend(collect_texts(source, child))

    if index.split('|')[-1].startswith('paragraph_'):
        return [' '.join(texts)]

    return texts


school_texts = collect_texts('md_example', 'md_example|school.md')
for text in school_texts:
    print('-', text)

**The other two Sources** hold the same documents in other formats. Both are converted to Markdown and then navigated exactly like
`md_example`:

- `mediawiki_example` (`xml_stream`) converts a MediaWiki XML export during the pilot. The current converter loses the text of links and
  bold words (they appear as `\1`), so the entity names are missing.
- `pdf_example` (`pdf_mirror`) converts each PDF the first time it is accessed. The default converter needs `pip install pdf-oxide`, which
  is not installed with mercury-graph. The reading order of the converted text may be wrong, which breaks the heading hierarchy.

For now, `md_example` is the reliable one.

## 8. Navigate the ontologies

Ontologies are navigated in the same way: `children_by_idx_<ontology>` for the hierarchy and `node_by_idx_<ontology>` for the properties
of a node. Indices start with the name of the ontology.

In [ ]:
print(run('children_by_idx_entities', index = 'entities'))
print(run('children_by_idx_entities', index = 'entities|place'))
print(run('children_by_idx_entities', index = 'entities|place|room'))
print(run('node_by_idx_entities', index = 'entities|place|room|laboratory'))

In [ ]:
print(run('children_by_idx_relationships', index = 'relationships|works_on'))
print(run('node_by_idx_relationships', index = 'relationships|works_on|leads'))

In `known_ids`, instances are navigated by type. Edges live under the special branch `_edge_`, and the index of an edge is
`known_ids|_edge_|<source>||<destination>||<edge id>`.

In [ ]:
print(run('children_by_idx_known_ids', index = 'known_ids|person|student'))
print(run('node_by_idx_known_ids',
          index = 'known_ids|_edge_|person|student|Noah Kim||project|Helios Solar Car||id_noah_leads_helios'))

## 9. Extract entities and relationships

The Formalizer proposes the entities (`hint_nodes_GLiNER2`) and relationships (`hint_edges_GLiNER2`) that it finds in a text, using the
ontologies as labels. The answer includes every label, so let's print only the ones with results.

In [ ]:
sentence = ('Elena Ruiz teaches science at Riverside High School. '
            'Noah Kim and Priya Shah study at Riverside High School.')

for label, found in run('hint_nodes_GLiNER2', text = sentence).items():
    if found:
        print('%-22s %s' % (label, found))

Relationships are grouped by the **definition** of the relationship type. Each candidate has its two ends, and some candidates come with
one end empty. This helper keeps the complete ones and maps each definition back to its identifier.

In [ ]:
relationship_ids = dict(zip(relationships['definition'], relationships['id']))


def complete_edges(text):
    """ Returns the relationships proposed for `text` that have both ends, as (source, relationship, destination) tuples. """

    edges = []
    for definition, candidates in run('hint_edges_GLiNER2', text = text).items():
        for candidate in candidates:
            (_, src), (_, dst) = candidate.items()
            for s in src:
                for d in dst:
                    edges.append((s, relationship_ids.get(definition, definition), d))

    return edges


complete_edges(sentence)

Until the Evidence graph can crawl the Sources by itself, you can do it by hand: take the texts collected in section 7 and run the
Formalizer on each of them.

In [ ]:
nodes = []
edges = []

for text in school_texts:
    for label, found in run('hint_nodes_GLiNER2', text = text).items():
        nodes.extend((entity, label, text) for entity in found)

    edges.extend(edge + (text,) for edge in complete_edges(text))

nodes = pd.DataFrame(nodes, columns = ['entity', 'type', 'text'])
nodes.groupby(['entity', 'type']).size().rename('mentions').reset_index()

In [ ]:
edges = pd.DataFrame(edges, columns = ['src', 'relationship', 'dst', 'text']).drop_duplicates()
edges

Read these tables as **candidates**, not facts. With an ontology that fits the documents, most of them are right, but typical mistakes are:

- relationship types proposed even when the text does not state them,
- the wrong subtype for a person (a student labeled as a teacher),
- the same relationship proposed more than once.

Keeping the `text` column is what makes the result traceable: every candidate points to the text it comes from, so it can be checked.

## 10. Talk to an agent

An agent exposes a single capability (`chat_with_<agent>`) with one argument, `content`. Its configuration (in `agents.jsonc`) sets the
model, the instructions (`you_are`, `you_must`) and the pieces it can use as tools. The `reader` agent can use `mediawiki_example`,
`GLiNER2` and `e_graph`.

This section needs Ollama running with the `qwen3:8b` model (see section 2).

In [ ]:
question = 'Who teaches at Riverside High School?'

answer = requests.post(BASE_URL + '/run',
                       json = {'name': 'chat_with_reader', 'arguments': {'content': question}},
                       timeout = 600).json()

print('finish_reason:', answer['finish_reason'])
print()
print(answer['message'].get('content'))

The agent does not call the tools itself. When the LLM asks for a tool, the **Endpoint** runs it, appends the result to the conversation and
calls the agent again, until the LLM gives a final answer. With `tool_call_history` enabled (the default), the answer includes that
conversation.

In [ ]:
for i, message in enumerate(answer.get('history', [])):
    for tool_call in message.get('tool_calls') or []:
        print('[%d] %-9s asks for %s %s' % (i, message['role'], tool_call['function']['name'], tool_call['function']['arguments']))

    if message['role'] != 'assistant':
        print('[%d] %-9s %s' % (i, message['role'], str(message.get('content'))[:200]))

The mechanism works, but **the answers are not reliable yet**. LLM answers vary from run to run, and this is what we have observed with
the template configuration:

- The `reader` instructions in the template are placeholders. The agent may run the Formalizer on the *question* instead of reading
  the documents, and then answer that it has no information.
- It may choose `crawl_e_graph`, which is not implemented yet. A failing tool ends the whole request with `finish_reason: "error"`.
- `max_tool_calls_per_query` (10 in `mge_endpoint.jsonc`) counts both tool calls and LLM calls, which limits how deep an agent can
  navigate a Source.

To go further, edit `you_are`, `you_must` and `tools` in `agents.jsonc` (for example, to use `md_example`). After any configuration
change, stop the server, pilot again and serve again.

Try your own questions: change `question` above and run the cells again.

## 11. Current limitations and next steps

What does not work yet:

- The Evidence graph (`e_graph`) cannot crawl the Sources yet (`crawl_e_graph`), so the graph is not built automatically.
- Extraction quality: the Formalizer proposes candidates that need to be checked (section 9).
- Agents run their tools correctly, but their answers are not reliable yet (section 10).
- Document converters: the MediaWiki conversion loses link and bold text, and PDF conversion needs an extra package and may break the
  reading order (section 7).

To use your own documents:

1. Point a Source to your files in `sources.jsonc` (`src_path`, `dst_path`, `type`). Relative paths are relative to the Endpoint folder.
2. Write an ontology for your domain, following the rules in section 4.
3. Give your agents instructions and tools that match your documents in `agents.jsonc`.
4. Stop the server, pilot and serve again.

When you finish, stop `mge serve` with `Ctrl+C` in its terminal. If a process is interrupted before releasing the lock, `mge unlock`
releases it.